# ⚽ Soccer Event Dataset Extractor
Extracts a structured CSV event dataset from an iPhone soccer video using YOLOv8 + ByteTrack.

**Runtime:** Make sure you're using a GPU runtime — *Runtime → Change runtime type → T4 GPU*

## Step 1 — Install Dependencies

In [ ]:
!pip install ultralytics opencv-python-headless numpy pandas scipy scikit-learn PyYAML -q

## Step 2 — Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# Clone or pull latest repo
if not os.path.exists('/content/soccer_analysis'):
    !git clone https://github.com/YOUR_USERNAME/soccer_analysis.git /content/soccer_analysis
else:
    !git -C /content/soccer_analysis pull

os.chdir('/content/soccer_analysis')
print('Working directory:', os.getcwd())

## Step 3 — Configure

In [ ]:
# ✏️  Edit these paths and settings
VIDEO_PATH = '/content/drive/MyDrive/soccer_videos/game.mp4'  # Path to your video on Drive
OUTPUT_CSV = '/content/drive/MyDrive/soccer_videos/events.csv'  # Where to save the output
CONFIG_PATH = 'config.yaml'  # Uses repo config.yaml by default

# Optional: Override config values here instead of editing config.yaml
# These will be written to a temporary config for this run.
import yaml

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ✏️ Define your goal regions and pitch boundary here (pixel coordinates)
# Use the calibration cell below to find the right values.
# cfg['goal_left']  = [0, 250, 60, 450]      # [x_min, y_min, x_max, y_max]
# cfg['goal_right'] = [1220, 250, 1280, 450]
# cfg['pitch_boundary'] = [50, 50, 1230, 670]

cfg['device'] = 'cuda'  # Force GPU on Colab

RUNTIME_CONFIG = '/tmp/colab_config.yaml'
with open(RUNTIME_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready:', cfg)

## Step 4 — (Optional) Calibration Helper
Run this cell to display the first frame and find goal/pitch pixel coordinates.

In [ ]:
import cv2
from IPython.display import display
from PIL import Image
import numpy as np

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = frame_rgb.shape[:2]
    print(f'Frame size: {w} x {h} pixels')
    display(Image.fromarray(frame_rgb))
else:
    print('Could not read frame from video. Check VIDEO_PATH.')

## Step 5 — Run Pipeline

In [ ]:
import sys
sys.path.insert(0, '/content/soccer_analysis')

from src.main import run

run(
    video_path=VIDEO_PATH,
    config_path=RUNTIME_CONFIG,
    output_path=OUTPUT_CSV,
)

## Step 6 — Inspect Results

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)
print(f'Total rows: {len(df)}')
print('\nEvent counts:')
print(df['event_type'].value_counts())
df.head(20)

## Step 7 — Visualize Events on Timeline

In [ ]:
import matplotlib.pyplot as plt

event_types = df[df['event_type'] != 'BALL_POSITION']['event_type'].unique()
fig, ax = plt.subplots(figsize=(16, 4))

colors = plt.cm.tab10.colors
for i, etype in enumerate(event_types):
    subset = df[df['event_type'] == etype]
    ax.scatter(subset['timestamp_sec'], [i] * len(subset), label=etype, color=colors[i % 10], s=20)

ax.set_yticks(range(len(event_types)))
ax.set_yticklabels(event_types)
ax.set_xlabel('Time (seconds)')
ax.set_title('Soccer Events Timeline')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## Step 8 — Ball Trajectory Plot

In [ ]:
ball_df = df[df['event_type'] == 'BALL_POSITION'].dropna(subset=['ball_x', 'ball_y'])

plt.figure(figsize=(12, 7))
plt.scatter(ball_df['ball_x'], ball_df['ball_y'],
            c=ball_df['timestamp_sec'], cmap='viridis', s=3, alpha=0.5)
plt.colorbar(label='Time (sec)')
plt.gca().invert_yaxis()  # Match image coordinate system
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')
plt.title('Ball Trajectory')
plt.tight_layout()
plt.show()